# Lesson 3: Environment Setup — Windows

**Week 1 · Data Engineering Course**

---

By the end of this notebook you will have:

- PostgreSQL 16 installed and running as a Windows service
- pgAdmin 4 installed and connected to your local database server
- A course database (`de_course`) and student user (`de_student`) ready to use
- Git installed and configured
- A GitHub account with a repository for your course files

**Time estimate:** 45–60 minutes

## What We're Installing

| Tool | What it is | Why we need it |
|------|-----------|----------------|
| **PostgreSQL 16** | The database server — stores and queries data | Our SQL learning environment and pipeline target |
| **pgAdmin 4** | A graphical interface for PostgreSQL | Browse tables, run queries, manage users — no command line required |
| **Git** | Version control software | Track changes to your SQL and notebooks |
| **GitHub** | Cloud platform for Git repositories | Back up your work and share it |
| **psycopg2** | Python library to connect to PostgreSQL | Used in later lessons to query data with Python |

PostgreSQL and pgAdmin are bundled in a **single installer** — you only need to download one file.
Git is a separate installer.

---

## Part 1: PostgreSQL & pgAdmin

---

## Step 1: Download the PostgreSQL Installer

1. Open your browser and go to:
   **https://www.postgresql.org/download/windows/**

2. Click **"Download the installer"** (takes you to the EDB download page)

3. Find **PostgreSQL 16** in the version table and click the **download icon** for **Windows x86-64**

   > The file is named something like `postgresql-16.x-windows-x64.exe`  
   > File size is approximately 300 MB

4. Save the file to your **Downloads** folder

---

## Step 2: Run the Installer

Right-click the downloaded `.exe` and choose **Run as administrator**. Click **Yes** on the UAC prompt.

---

### Screen 1 — Welcome
Click **Next**.

---

### Screen 2 — Installation Directory
Leave the default:
```
C:\Program Files\PostgreSQL\16
```
Click **Next**.

---

### Screen 3 — Select Components

Make sure **all four** boxes are checked:

| Component | Keep? |
|-----------|-------|
| ✅ PostgreSQL Server | Yes — this is the database engine |
| ✅ pgAdmin 4 | Yes — this is the GUI we'll use |
| ✅ Stack Builder | Yes (default) |
| ✅ Command Line Tools | Yes — includes `psql` |

Click **Next**.

---

### Screen 4 — Data Directory
Leave the default:
```
C:\Program Files\PostgreSQL\16\data
```
Click **Next**.

---

### Screen 5 — Password ⚠️

This sets the password for the **`postgres` superuser**.

- Choose a password you will remember (e.g. `postgres123`)
- **Write it down now** — you need it every time you connect
- Confirm the password in the second field

Click **Next**.

---

### Screen 6 — Port
Leave the default: **5432**. Click **Next**.

---

### Screen 7 — Locale
Leave as **[Default locale]**. Click **Next**.

---

### Screen 8 — Pre Installation Summary
Review and click **Next** to begin installation.

---

### Screen 9 — Installing...
Wait 1–3 minutes for the progress bar to complete.

---

### Screen 10 — Finish
- **Uncheck** "Launch Stack Builder at exit"
- Click **Finish**

---

## Step 3: Verify PostgreSQL is Running

PostgreSQL installs itself as a **Windows Service** that starts automatically on boot.

### Check via Services
1. Press `Win + R`, type `services.msc`, press Enter
2. Find **postgresql-x64-16** in the list
3. Status should be **Running**, Startup Type should be **Automatic**

If it shows Stopped, right-click → **Start**.

---

### Quick test with SQL Shell

Open **Start → PostgreSQL 16 → SQL Shell (psql)**

Press **Enter** to accept each default (Server, Database, Port, Username), then enter your `postgres` password.

You should see:
```
postgres=#
```

Type `\q` and press Enter to exit. If you see this prompt, PostgreSQL is working.

In [ ]:
import subprocess

result = subprocess.run(["psql", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ psql found:", result.stdout.strip())
else:
    print("psql not found in PATH — this is OK.")
    print("We will use pgAdmin and Python instead.")
    print("To add psql to PowerShell PATH:")
    print("  C:\\Program Files\\PostgreSQL\\16\\bin")

---

## Step 4: Open pgAdmin 4

1. Open the **Start menu** and search for **pgAdmin 4**
2. Click to launch — pgAdmin opens as a **tab in your default browser**
   (it runs a local web server; the URL looks like `http://127.0.0.1:51723`)

---

### Set the pgAdmin Master Password

The first time pgAdmin opens, it asks for a **master password**.

- This protects your saved server connections inside pgAdmin
- It is **separate** from the PostgreSQL `postgres` password set during installation
- Choose something memorable (e.g. `pgadmin123`) and click **OK**

> You will enter this password each time you open pgAdmin.

---

## Step 5: Connect pgAdmin to Your Local Server

In the left panel you should already see **Servers → PostgreSQL 16** (the installer registers it automatically).

1. Click the **arrow** next to **Servers** to expand it
2. Click **PostgreSQL 16**
3. A dialog appears: **"Please enter the password for the user 'postgres'"**
4. Enter the password you set in Step 2
5. Optionally check **"Save Password"**
6. Click **OK**

The server expands and you will see:

```
PostgreSQL 16
├── Databases
│   ├── postgres
│   ├── template0
│   └── template1
├── Login/Group Roles
└── Tablespaces
```

You are connected!

---

## Step 6: Create the Course Database

We'll create a database named **`de_course`** to hold all course data.

### Option A — pgAdmin GUI

1. Right-click **Databases** (under PostgreSQL 16 in the left panel)
2. Choose **Create → Database...**
3. In the **General** tab set **Database:** `de_course`
4. Click **Save**

`de_course` now appears under Databases.

---

### Option B — Query Tool

1. Click **Tools → Query Tool**
2. Make sure the connection shows `postgres` at the top
3. Type:
```sql
CREATE DATABASE de_course;
```
4. Press **F5** to run
5. You'll see `CREATE DATABASE` in the Messages tab

---

## Step 7: Create the Student User

We'll create a dedicated user **`de_student`** for this course instead of using the `postgres` superuser — this mirrors real-world best practice.

### Option A — pgAdmin GUI

1. Right-click **Login/Group Roles** → **Create → Login/Group Role...**

**General tab:**
- Name: `de_student`

**Definition tab:**
- Password: `learning123`

**Privileges tab:**
- Can login? → **Yes**

2. Click **Save**

---

### Option B — Query Tool

Open **Tools → Query Tool**, connect to `de_course`, and run:

```sql
CREATE USER de_student WITH PASSWORD 'learning123';
GRANT ALL PRIVILEGES ON DATABASE de_course TO de_student;
```

Press **F5**. You'll see `CREATE ROLE` and `GRANT` in the Messages tab.

---

## Step 8: Create Database & User via Python (Optional)

If you prefer Python over the pgAdmin GUI, run the cell below.  
**Skip this if you already completed Steps 6–7 in pgAdmin.**

In [ ]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# Replace with the password you set during installation (Step 2, Screen 5)
POSTGRES_PASSWORD = "postgres123"

try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="postgres",
        user="postgres",
        password=POSTGRES_PASSWORD
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    for stmt in [
        "CREATE DATABASE de_course",
        "CREATE USER de_student WITH PASSWORD 'learning123'",
        "GRANT ALL PRIVILEGES ON DATABASE de_course TO de_student",
    ]:
        try:
            cur.execute(stmt)
            print(f"✓  {stmt}")
        except psycopg2.errors.DuplicateDatabase:
            print(f"   (already exists)  {stmt}")
        except psycopg2.errors.DuplicateObject:
            print(f"   (already exists)  {stmt}")

    cur.close()
    conn.close()
    print("\nDone!")

except psycopg2.OperationalError as e:
    print(f"Connection failed: {e}")
    print("Check that POSTGRES_PASSWORD matches what you set during installation.")

---

## Step 9: Explore pgAdmin

### Run your first query

1. In the left panel expand **de_course → Schemas → public**
2. Click **Tools → Query Tool**
3. Type and press **F5**:

```sql
SELECT version();
```

You'll see the PostgreSQL version string in the **Data Output** tab.

---

### pgAdmin layout

```
┌─────────────────────────────────────────────────────────────┐
│  Menu: File · Object · Tools · Help                         │
├──────────────────┬──────────────────────────────────────────┤
│  Object          │  Query Tool (SQL editor)                 │
│  Explorer        │  ┌────────────────────────────────────┐  │
│                  │  │  SELECT version();                 │  │
│  Servers         │  └────────────────────────────────────┘  │
│  └ PostgreSQL 16 │  ▶ Execute (F5)                          │
│    └ Databases   ├──────────────────────────────────────────┤
│      └ de_course │  Data Output │ Messages                  │
│        └ Schemas │  Results appear here                     │
└──────────────────┴──────────────────────────────────────────┘
```

| Area | What you do there |
|------|------------------|
| **Object Explorer** | Navigate databases, tables, users |
| **Query Tool** | Write and run SQL |
| **Data Output** | See query results as a table |
| **Messages** | See success / error messages |

---

### Browse table data without SQL

After loading data in the lab:
1. Expand `de_course → Schemas → public → Tables`
2. Right-click a table → **View/Edit Data → First 100 Rows**

pgAdmin runs the SELECT automatically and shows a spreadsheet view.

---

## Step 10: Verify the Python Connection (Optional)

Run this to confirm Python can talk to your PostgreSQL database.  
First install the driver if needed:

In [ ]:
# Run this cell first if psycopg2 is not installed
# !pip install psycopg2-binary

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="de_course",
        user="de_student",
        password="learning123"
    )
    cur = conn.cursor()
    cur.execute("SELECT current_database(), current_user;")
    db, user = cur.fetchone()
    print(f"✓ Connected to database '{db}' as user '{user}'")
    cur.close()
    conn.close()

except psycopg2.OperationalError as e:
    print(f"✗ Connection failed: {e}")
    print()
    print("Common causes:")
    print("  1. PostgreSQL service not running — check services.msc")
    print("  2. de_student user not created — complete Step 7")
    print("  3. de_course database not created — complete Step 6")

---

## pgAdmin Quick Reference

### Keyboard shortcuts in the Query Tool

| Shortcut | Action |
|----------|--------|
| `F5` | Run the full query |
| `Shift+F5` | Run selected text only |
| `F7` | Explain (show query plan) |
| `Ctrl+S` | Save the SQL file |
| `Ctrl+/` | Comment / uncomment lines |
| `Ctrl+Z` | Undo |

### Common tasks

| Task | How |
|------|-----|
| Create a table | Right-click Tables → Create → Table, or `CREATE TABLE` in Query Tool |
| View table data | Right-click table → View/Edit Data → First 100 Rows |
| Describe a table | Right-click table → Properties → Columns tab |
| Drop a table | Right-click table → Delete/Drop |
| Export results to CSV | Run query → click the Download icon above the results grid |
| Run a .sql file | Query Tool → File → Open → select file → F5 |

---

## Troubleshooting — PostgreSQL

**pgAdmin won't open / browser shows a connection error**  
The local server takes a few seconds to start. Wait 15 seconds and refresh the browser tab. If still broken, relaunch pgAdmin from the Start menu.

---

**"FATAL: password authentication failed for user postgres"**  
You entered the wrong password. This is the password from Screen 5 of the installer. If forgotten, ask your instructor about resetting via `pg_hba.conf`.

---

**"Could not connect to server: Connection refused"**  
PostgreSQL isn't running:
1. Press `Win + R` → type `services.msc` → Enter
2. Find `postgresql-x64-16` → right-click → **Start**
3. Try pgAdmin again

---

**`psycopg2.OperationalError` in Python even though pgAdmin works**  
- Make sure `host="localhost"` and `port=5432`
- Run `pip install psycopg2-binary` in your Python environment
- Double-check the username (`de_student`) and password (`learning123`)

---

## Part 1 Summary

| Item | Value |
|------|-------|
| PostgreSQL version | 16 |
| Installation path | `C:\Program Files\PostgreSQL\16\` |
| Windows service name | `postgresql-x64-16` |
| Default port | `5432` |
| Superuser | `postgres` (password: what you set during install) |
| Course database | `de_course` |
| Course user | `de_student` / `learning123` |
| pgAdmin | Opens in browser at `http://127.0.0.1:<port>` |

---

## Part 2: Git & GitHub Setup

**Git** is version control software — it tracks every change you make to your files so you can see history, undo mistakes, and collaborate with others. Every data engineering team uses Git.

**GitHub** is the cloud platform where you store and share Git repositories.

By the end of Part 2 you will have:
- Git installed and your name/email configured
- A GitHub account
- A course repository connected between your computer and GitHub

---

## Step 11: Download Git for Windows

1. Open your browser and go to: **https://git-scm.com/download/win**
2. The 64-bit installer downloads automatically
   (filename: `Git-2.x.x-64-bit.exe`, ~60 MB)
3. Save the file to your **Downloads** folder

---

## Step 12: Install Git

Run the downloaded `.exe`. Most screens can be left at their defaults — note these three:

---

### Choosing a default editor
The installer asks which editor Git should use for commit messages.  
Select **Notepad** from the dropdown (simplest choice on Windows).

---

### Adjusting PATH environment ⚠️ (important)
Select: **"Git from the command line and also from 3rd-party software"**  
This lets you run `git` from Command Prompt, PowerShell, and Jupyter.

---

### Line ending conversions
Leave the default: **"Checkout Windows-style, commit Unix-style line endings"**

---

All other screens: accept the defaults → **Next** → **Install** → **Finish**.

Git Bash (a Unix-style terminal for Windows) is installed alongside Git — you'll use it in the next steps.

In [ ]:
import subprocess

result = subprocess.run(["git", "--version"], capture_output=True, text=True, shell=True)
if result.returncode == 0:
    print("✓ Git is installed:", result.stdout.strip())
else:
    print("✗ Git not found.")
    print("  Make sure you selected 'Git from the command line' during installation.")
    print("  You may need to restart Jupyter after installing Git.")

---

## Step 13: Configure Your Identity

Git attaches your name and email to every commit you make. This is how teammates know who made a change.

Open **Git Bash** (search for it in the Start menu) and run these two commands — replace with your own name and email:

```bash
git config --global user.name "Your Name"
git config --global user.email "your@email.com"
```

**Verify the settings were saved:**

```bash
git config --list
```

You should see `user.name=...` and `user.email=...` in the output.

> Use the same email you'll register with GitHub in the next step.

In [ ]:
import subprocess

result = subprocess.run(["git", "config", "--list"], capture_output=True, text=True, shell=True)
if result.returncode == 0:
    identity_lines = [l for l in result.stdout.strip().split('\n') if l.startswith('user.')]
    if identity_lines:
        print("✓ Git identity configured:")
        for line in identity_lines:
            print(" ", line)
    else:
        print("Git found but user identity not configured.")
        print("Open Git Bash and run the git config commands in Step 13.")
else:
    print("✗ Git not found — complete Steps 11–12 first.")

---

## Step 14: Create a GitHub Account

If you don't already have one:

1. Go to **https://github.com**
2. Click **Sign up**
3. Enter a username, the same email you used in Step 13, and a password
4. Complete the verification puzzle
5. Choose the **Free** plan
6. Check your email and click the verification link

> Your GitHub username will appear in your repository URLs and on your public profile.

---

## Step 15: Create Your Course Repository on GitHub

A **repository** (repo) is a folder tracked by Git. We'll create one on GitHub to back up all your course work.

### Create the repo

1. Log in to **https://github.com**
2. Click the **+** icon (top right) → **New repository**
3. Fill in:

   | Field | Value |
   |-------|-------|
   | Repository name | `data-engineering-course` |
   | Description | My Data Engineering coursework |
   | Visibility | **Private** (your in-progress work) |
   | Initialize with README | ✅ Yes |

4. Click **Create repository**

### Copy the clone URL

1. On the new repository page, click the green **Code** button
2. Make sure **HTTPS** is selected
3. Copy the URL — it looks like:
   `https://github.com/yourusername/data-engineering-course.git`

---

## Step 16: Connect Your Course Files to GitHub

Open **Git Bash** and run the following commands one at a time.
Replace `YourName` with your Windows username and the GitHub URL with your own.

```bash
# Navigate to your lessons folder
cd C:/Users/YourName/lessons

# Initialize a Git repository
git init

# Stage all files
git add .

# Create your first commit
git commit -m "Add Week 1 course materials"

# Connect to your GitHub repository
git remote add origin https://github.com/yourusername/data-engineering-course.git

# Push to GitHub (only needed the first time)
git push -u origin main
```

Refresh your GitHub repository page — you should see the `week1/` folder appear.

---

### Basic Git workflow going forward

After completing each notebook, run:

```bash
git add week1/lesson4_intro_to_sql.ipynb
git commit -m "Complete Lesson 4 exercises"
git push
```

| Command | What it does |
|---------|-------------|
| `git add <file>` | Stage a file to be committed |
| `git add .` | Stage all changed files |
| `git commit -m "message"` | Save a snapshot with a description |
| `git push` | Upload commits to GitHub |
| `git status` | Show which files have changed |
| `git log` | Show the history of commits |

---

## Troubleshooting — Git & GitHub

**`git` is not recognized as a command in PowerShell**  
Git was not added to PATH during installation. Uninstall Git and reinstall, making sure to select **"Git from the command line and also from 3rd-party software"** on the PATH screen. Alternatively, use **Git Bash** instead of PowerShell.

---

**`git push` asks for a username and password**  
GitHub no longer accepts passwords for `git push`. You need a **Personal Access Token (PAT)**:  
1. GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)  
2. Generate new token → check **repo** scope → copy the token  
3. Use the token as your password when Git asks

---

**`error: remote origin already exists`**  
The remote was already added. Update it instead:
```bash
git remote set-url origin https://github.com/yourusername/data-engineering-course.git
```

---

**`git push` rejected — non-fast-forward**  
The GitHub repo has commits your local copy doesn't. Pull first:
```bash
git pull origin main --allow-unrelated-histories
git push
```

---

## Summary

### PostgreSQL

| Item | Value |
|------|-------|
| Version | PostgreSQL 16 |
| Service name | `postgresql-x64-16` |
| Port | `5432` |
| Superuser | `postgres` |
| Course database | `de_course` |
| Course user | `de_student` / `learning123` |

### Git & GitHub

| Item | Value |
|------|-------|
| Git location | `C:\Program Files\Git\` |
| Git Bash | Start menu → Git Bash |
| Your repo URL | `https://github.com/<yourusername>/data-engineering-course` |
| Daily workflow | `git add` → `git commit` → `git push` |

---

You're ready for the Lab — proceed to `labs/lab1_load_csv.ipynb`.